# Kaggle Roboflow Workflow

This notebook bundles three workflows into one Kaggle-ready file:

1. Download a Roboflow dataset version and clean invalid YOLO rows.
2. Sample suspicious images for targeted manual review.
3. Mine false positives and false negatives after training.

It is designed for fire/smoke object detection datasets exported from Roboflow in YOLO format.

In [ ]:
!pip install -qU roboflow ultralytics pyyaml pandas matplotlib opencv-python-headless

## Configuration

Recommended: create a Kaggle secret named `ROBOFLOW_API_KEY` and set `USE_KAGGLE_SECRET = True`.

If you prefer, you can set `USE_KAGGLE_SECRET = False` and paste the key directly into the cell.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import random
import hashlib

import cv2
import requests
import zipfile
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from roboflow import Roboflow
from ultralytics import YOLO

USE_KAGGLE_SECRET = True

if USE_KAGGLE_SECRET:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
else:
    ROBOFLOW_API_KEY = "PASTE_YOUR_PRIVATE_ROBOFLOW_API_KEY_HERE"

WORKSPACE_ID = "your-workspace-id"
PROJECT_ID = "your-project-id"
VERSION_NUMBER = 1
EXPORT_FORMAT = "yolov8"

DOWNLOAD_DIR = Path("/kaggle/working/roboflow_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/kaggle/working/roboflow_export.zip")
EXTRACT_DIR = Path("/kaggle/working/roboflow_export")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Download Dataset Version From Roboflow

In [ ]:
def normalize_names(names):
    if isinstance(names, list):
        return {i: name for i, name in enumerate(names)}
    return {int(k): v for k, v in names.items()}

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_ID).project(PROJECT_ID)
version = project.version(VERSION_NUMBER)

export_info_url = f"https://api.roboflow.com/{WORKSPACE_ID}/{PROJECT_ID}/{VERSION_NUMBER}/{EXPORT_FORMAT}?api_key={ROBOFLOW_API_KEY}"
resp = requests.get(export_info_url, timeout=120)
resp.raise_for_status()
payload = resp.json()
download_url = payload["export"]["link"]

with requests.get(download_url, stream=True, timeout=600) as r:
    r.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

raw_root = EXTRACT_DIR
print("Downloaded ZIP:", ZIP_PATH)
print("Extracted to:", raw_root)

split_hits = defaultdict(set)
for split in ["train", "valid", "val", "test"]:
    pattern = f"{split}/images"
    for img_dir in raw_root.rglob(pattern):
        root = img_dir.parent.parent
        label_dir = root / split / "labels"
        if label_dir.exists():
            split_hits[root].add(split)

if split_hits:
    ranked_roots = sorted(split_hits.items(), key=lambda item: (-len(item[1]), len(str(item[0]))))
    DATASET_ROOT = ranked_roots[0][0]
else:
    child_dirs = [p for p in raw_root.iterdir() if p.is_dir()]
    if len(child_dirs) == 1:
        DATASET_ROOT = child_dirs[0]
    else:
        print("Top-level extracted contents:")
        for p in child_dirs:
            print("-", p)
        raise FileNotFoundError(f"Could not detect dataset root under {raw_root}")

print("Detected dataset root:", DATASET_ROOT)
print("Detected split coverage:")
for root, splits in sorted(split_hits.items(), key=lambda item: str(item[0])):
    print("-", root, sorted(splits))

SOURCE_DATA_YAML = DATASET_ROOT / "data.yaml"
if SOURCE_DATA_YAML.exists():
    source_cfg = yaml.safe_load(SOURCE_DATA_YAML.read_text())
    source_names = source_cfg.get("names", {0: "smoke", 1: "fire"})
else:
    source_names = {0: "smoke", 1: "fire"}

CLASS_NAMES = normalize_names(source_names)
ALLOWED_CLASSES = set(CLASS_NAMES.keys())

DATA_YAML = Path("/kaggle/working/data.yaml")
generated_cfg = {
    "path": str(DATASET_ROOT),
    "train": "train/images" if (DATASET_ROOT / "train" / "images").exists() else None,
    "val": "valid/images" if (DATASET_ROOT / "valid" / "images").exists() else ("val/images" if (DATASET_ROOT / "val" / "images").exists() else None),
    "test": "test/images" if (DATASET_ROOT / "test" / "images").exists() else None,
    "names": CLASS_NAMES,
}
generated_cfg = {k: v for k, v in generated_cfg.items() if v is not None}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(generated_cfg, f, sort_keys=False)

print("Dataset root:", DATASET_ROOT)
print("Using data.yaml:", DATA_YAML)
print("Class names:", CLASS_NAMES)
print(DATA_YAML.read_text())

## 2. Clean Invalid YOLO Rows

This pass removes rows that are clearly invalid:

- wrong column count
- parse errors
- invalid class IDs
- coordinates outside `[0, 1]`
- non-positive width or height

Empty label files are kept. They represent background images.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def existing_split_names(root: Path):
    splits = []
    for name in ["train", "valid", "val", "test"]:
        if (root / name / "images").exists() and (root / name / "labels").exists():
            splits.append(name)
    return splits

def clean_label_file(label_path: Path):
    raw_lines = label_path.read_text(encoding="utf-8").splitlines()
    kept = []
    removed = []

    for line_no, raw in enumerate(raw_lines, 1):
        line = raw.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            removed.append((line_no, line, "wrong_column_count"))
            continue

        try:
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except Exception:
            removed.append((line_no, line, "parse_error"))
            continue

        if cls not in ALLOWED_CLASSES:
            removed.append((line_no, line, "invalid_class"))
            continue

        if any(v < 0 or v > 1 for v in (x, y, w, h)):
            removed.append((line_no, line, "coords_out_of_bounds"))
            continue

        if w <= 0 or h <= 0:
            removed.append((line_no, line, "non_positive_size"))
            continue

        kept.append(f"{cls} {x} {y} {w} {h}")

    label_path.write_text("\n".join(kept), encoding="utf-8")
    return kept, removed

def summarize_split(root: Path, split_name: str):
    img_dir = root / split_name / "images"
    lbl_dir = root / split_name / "labels"

    image_count = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
    label_count = sum(1 for _ in lbl_dir.glob("*.txt"))
    empty_labels = 0

    for txt in lbl_dir.glob("*.txt"):
        if txt.read_text(encoding="utf-8").strip() == "":
            empty_labels += 1

    return {
        "split": split_name,
        "images": image_count,
        "label_files": label_count,
        "empty_label_files": empty_labels,
    }

SPLITS = existing_split_names(DATASET_ROOT)
print("Detected splits:", SPLITS)

stats = Counter()
bad_examples = []

for split in SPLITS:
    label_dir = DATASET_ROOT / split / "labels"
    for label_path in label_dir.glob("*.txt"):
        stats["files_seen"] += 1
        before_lines = [x for x in label_path.read_text(encoding="utf-8").splitlines() if x.strip()]
        kept, removed = clean_label_file(label_path)

        stats["rows_before"] += len(before_lines)
        stats["rows_after"] += len(kept)
        stats["rows_removed"] += len(removed)

        if removed:
            stats["files_modified"] += 1
            for _, _, reason in removed:
                stats[f"removed_{reason}"] += 1
            if len(bad_examples) < 20:
                bad_examples.append({
                    "file": str(label_path),
                    "removed": removed[:5],
                })

print("Cleaning complete.")
print(dict(stats))

print("\nSample cleaned files:")
for item in bad_examples[:10]:
    print(item["file"])
    for row in item["removed"]:
        print(" ", row)

summary_rows = [summarize_split(DATASET_ROOT, split) for split in SPLITS]
summary_df = pd.DataFrame(summary_rows)
summary_df

## 3. Build Suspicious-Image Review Buckets

This section creates compact review manifests so you can inspect the most valuable subset instead of the entire dataset.

In [ ]:
def parse_label_file(label_path: Path):
    rows = []
    if not label_path.exists():
        return rows

    for line in label_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            continue
        cls, x, y, w, h = parts
        rows.append({
            "cls": int(cls),
            "x": float(x),
            "y": float(y),
            "w": float(w),
            "h": float(h),
            "area": float(w) * float(h),
            "aspect": (float(w) / float(h)) if float(h) > 0 else np.inf,
        })
    return rows

records = []
for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    for img_path in img_dir.iterdir():
        if img_path.suffix.lower() not in IMG_EXTS:
            continue

        label_path = lbl_dir / f"{img_path.stem}.txt"
        rows = parse_label_file(label_path)

        smoke_count = sum(1 for r in rows if CLASS_NAMES.get(r["cls"], str(r["cls"])).lower() == "smoke")
        fire_count = sum(1 for r in rows if CLASS_NAMES.get(r["cls"], str(r["cls"])).lower() == "fire")
        box_count = len(rows)
        areas = [r["area"] for r in rows]
        aspects = [r["aspect"] for r in rows]

        records.append({
            "split": split,
            "image_name": img_path.name,
            "image_path": str(img_path),
            "label_path": str(label_path),
            "box_count": box_count,
            "smoke_count": smoke_count,
            "fire_count": fire_count,
            "is_background": int(box_count == 0),
            "min_area": min(areas) if areas else np.nan,
            "max_area": max(areas) if areas else np.nan,
            "mean_area": float(np.mean(areas)) if areas else np.nan,
            "max_aspect": max(aspects) if aspects else np.nan,
            "min_aspect": min(aspects) if aspects else np.nan,
        })

df = pd.DataFrame(records)
print("Total images:", len(df))
display(df.groupby("split").agg(
    images=("image_name", "count"),
    backgrounds=("is_background", "sum"),
    total_boxes=("box_count", "sum"),
    smoke_boxes=("smoke_count", "sum"),
    fire_boxes=("fire_count", "sum"),
))

tiny_thresh = 0.0025
huge_thresh = 0.50
many_boxes_thresh = 6
extreme_aspect_thresh = 6.0

bucket_background = df[df["is_background"] == 1].copy()
bucket_many = df[df["box_count"] >= many_boxes_thresh].copy()
bucket_tiny = df[(df["box_count"] > 0) & (df["min_area"] < tiny_thresh)].copy()
bucket_huge = df[(df["box_count"] > 0) & (df["max_area"] > huge_thresh)].copy()
bucket_extreme_aspect = df[
    (df["box_count"] > 0) &
    ((df["max_aspect"] > extreme_aspect_thresh) | (df["min_aspect"] < 1 / extreme_aspect_thresh))
].copy()
bucket_fire_only = df[(df["fire_count"] > 0) & (df["smoke_count"] == 0)].copy()
bucket_smoke_only = df[(df["smoke_count"] > 0) & (df["fire_count"] == 0)].copy()
bucket_both = df[(df["smoke_count"] > 0) & (df["fire_count"] > 0)].copy()

def sample_bucket(bucket_df, n=40):
    if len(bucket_df) == 0:
        return bucket_df
    return bucket_df.sample(n=min(n, len(bucket_df)), random_state=SEED)

review_sets = {
    "background_sample": sample_bucket(bucket_background, 80),
    "many_boxes_sample": sample_bucket(bucket_many, 60),
    "tiny_boxes_sample": sample_bucket(bucket_tiny, 60),
    "huge_boxes_sample": sample_bucket(bucket_huge, 40),
    "extreme_aspect_sample": sample_bucket(bucket_extreme_aspect, 40),
    "fire_only_sample": sample_bucket(bucket_fire_only, 40),
    "smoke_only_sample": sample_bucket(bucket_smoke_only, 40),
    "both_classes_sample": sample_bucket(bucket_both, 40),
}

priority_review = pd.concat([
    sample_bucket(bucket_background, 60),
    sample_bucket(bucket_tiny, 40),
    sample_bucket(bucket_many, 30),
    sample_bucket(bucket_both, 30),
    sample_bucket(bucket_huge, 20),
]).drop_duplicates(subset=["image_path"]).reset_index(drop=True)

REVIEW_DIR = Path("/kaggle/working/review_manifests")
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

for name, subset in review_sets.items():
    out_csv = REVIEW_DIR / f"{name}.csv"
    subset.to_csv(out_csv, index=False)
    print(f"Saved {name}: {out_csv} ({len(subset)} rows)")

priority_csv = REVIEW_DIR / "priority_review.csv"
priority_review.to_csv(priority_csv, index=False)
print("Saved priority review:", priority_csv, len(priority_review))

In [ ]:
def draw_gt_boxes(image_path, label_path):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    if Path(label_path).exists():
        for row in parse_label_file(Path(label_path)):
            cls = row["cls"]
            xc, yc, bw, bh = row["x"], row["y"], row["w"], row["h"]
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            class_name = CLASS_NAMES.get(cls, str(cls))
            color = (0, 170, 255) if class_name.lower() == "smoke" else (255, 80, 80)

            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(image, class_name, (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return image

def show_review_subset(subset_df, title="", n=12, cols=3):
    if len(subset_df) == 0:
        print("No images in this subset.")
        return

    subset_df = subset_df.head(n)
    rows = math.ceil(len(subset_df) / cols)

    plt.figure(figsize=(5 * cols, 4.5 * rows))
    for i, (_, row) in enumerate(subset_df.iterrows(), 1):
        vis = draw_gt_boxes(row["image_path"], row["label_path"])
        plt.subplot(rows, cols, i)
        plt.imshow(vis)
        plt.title(
            f'{row["split"]} | {row["image_name"]}\n'
            f'boxes={row["box_count"]} smoke={row["smoke_count"]} fire={row["fire_count"]}'
        )
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

show_review_subset(priority_review, "Priority Review Set", n=18, cols=3)
show_review_subset(review_sets["background_sample"], "Background Sample", n=12, cols=3)
show_review_subset(review_sets["tiny_boxes_sample"], "Tiny Boxes Sample", n=12, cols=3)

## 4. Optional Training Cell

Leave `RUN_TRAINING = False` until you are ready.

This uses the cleaned Roboflow export directly.

In [ ]:
RUN_TRAINING = False

MODEL_NAME = "yolo26n.pt"
RUN_NAME = "roboflow_fire_smoke_yolo26n"
EPOCHS = 20
IMGSZ = 512
BATCH = 16
TRAIN_FRACTION = 0.2

if RUN_TRAINING:
    model = YOLO(MODEL_NAME)
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=4,
        cache="disk",
        pretrained=True,
        optimizer="auto",
        cos_lr=True,
        patience=10,
        close_mosaic=10,
        project="/kaggle/working/runs/detect",
        name=RUN_NAME,
        exist_ok=True,
        seed=SEED,
        deterministic=False,
        amp=False,
        save=True,
        val=True,
        plots=True,
        fraction=TRAIN_FRACTION,
    )
    print("Run directory:", results.save_dir)

## 5. Post-Training Error Mining

This section runs a trained model over one split and creates manifests for:

- images with many false positives
- images with many false negatives
- images with both

It uses greedy IoU matching at the image level.

In [ ]:
def yolo_to_xyxy(row):
    x, y, w, h = row["x"], row["y"], row["w"], row["h"]
    return np.array([x - w / 2, y - h / 2, x + w / 2, y + h / 2], dtype=float)

def iou_xyxy(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, xa2 - xa1) * max(0.0, ya2 - ya1)
    area_b = max(0.0, xb2 - xb1) * max(0.0, yb2 - yb1)
    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0
    return inter_area / union

def gt_boxes_for_image(label_path: Path):
    rows = parse_label_file(label_path)
    gt = []
    for row in rows:
        gt.append({
            "cls": row["cls"],
            "xyxy": yolo_to_xyxy(row),
        })
    return gt

def pred_boxes_for_result(result, image_shape):
    h, w = image_shape[:2]
    preds = []

    if result.boxes is None or len(result.boxes) == 0:
        return preds

    xyxy = result.boxes.xyxy.cpu().numpy()
    conf = result.boxes.conf.cpu().numpy()
    cls = result.boxes.cls.cpu().numpy().astype(int)

    for b, c, s in zip(xyxy, cls, conf):
        preds.append({
            "cls": int(c),
            "conf": float(s),
            "xyxy": np.array([b[0] / w, b[1] / h, b[2] / w, b[3] / h], dtype=float),
        })
    return preds

def greedy_match(preds, gts, iou_threshold=0.5):
    matched_pred = set()
    matched_gt = set()
    candidates = []

    for pi, pred in enumerate(preds):
        for gi, gt in enumerate(gts):
            if pred["cls"] != gt["cls"]:
                continue
            iou = iou_xyxy(pred["xyxy"], gt["xyxy"])
            if iou >= iou_threshold:
                candidates.append((iou, pred.get("conf", 0.0), pi, gi))

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)

    for _, _, pi, gi in candidates:
        if pi in matched_pred or gi in matched_gt:
            continue
        matched_pred.add(pi)
        matched_gt.add(gi)

    return matched_pred, matched_gt

def chunked(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def run_error_mining(model, split_name, conf=0.25, imgsz=512, batch_size=32, iou_threshold=0.5):
    img_dir = DATASET_ROOT / split_name / "images"
    lbl_dir = DATASET_ROOT / split_name / "labels"
    image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS])

    records = []
    for batch in chunked(image_paths, batch_size):
        predictions = model.predict(
            source=[str(p) for p in batch],
            conf=conf,
            imgsz=imgsz,
            device=0 if torch.cuda.is_available() else "cpu",
            verbose=False,
        )

        for img_path, pred in zip(batch, predictions):
            image = cv2.imread(str(img_path))
            gt = gt_boxes_for_image(lbl_dir / f"{img_path.stem}.txt")
            pred_boxes = pred_boxes_for_result(pred, image.shape)

            matched_pred, matched_gt = greedy_match(pred_boxes, gt, iou_threshold=iou_threshold)
            fp_count = len(pred_boxes) - len(matched_pred)
            fn_count = len(gt) - len(matched_gt)

            records.append({
                "split": split_name,
                "image_name": img_path.name,
                "image_path": str(img_path),
                "label_path": str(lbl_dir / f"{img_path.stem}.txt"),
                "gt_count": len(gt),
                "pred_count": len(pred_boxes),
                "fp_count": fp_count,
                "fn_count": fn_count,
                "matched_count": len(matched_gt),
            })

    error_df = pd.DataFrame(records)
    out_dir = Path("/kaggle/working/error_mining") / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    error_df.sort_values(["fp_count", "fn_count", "pred_count"], ascending=False).to_csv(out_dir / "all_errors.csv", index=False)

    top_fp = error_df.sort_values(["fp_count", "fn_count", "pred_count"], ascending=False).query("fp_count > 0").head(100)
    top_fn = error_df.sort_values(["fn_count", "fp_count", "gt_count"], ascending=False).query("fn_count > 0").head(100)
    top_both = error_df.sort_values(["fp_count", "fn_count"], ascending=False).query("fp_count > 0 and fn_count > 0").head(100)

    top_fp.to_csv(out_dir / "top_false_positives.csv", index=False)
    top_fn.to_csv(out_dir / "top_false_negatives.csv", index=False)
    top_both.to_csv(out_dir / "top_mixed_errors.csv", index=False)

    return error_df, top_fp, top_fn, top_both, out_dir

def draw_gt_and_pred(image_path, pred_result, label_path):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    for row in parse_label_file(Path(label_path)):
        cls = row["cls"]
        x1 = int((row["x"] - row["w"] / 2) * w)
        y1 = int((row["y"] - row["h"] / 2) * h)
        x2 = int((row["x"] + row["w"] / 2) * w)
        y2 = int((row["y"] + row["h"] / 2) * h)
        label = f"GT {CLASS_NAMES.get(cls, cls)}"
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, label, (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    if pred_result.boxes is not None and len(pred_result.boxes) > 0:
        xyxy = pred_result.boxes.xyxy.cpu().numpy()
        conf = pred_result.boxes.conf.cpu().numpy()
        cls = pred_result.boxes.cls.cpu().numpy().astype(int)
        for b, c, s in zip(xyxy, cls, conf):
            x1, y1, x2, y2 = map(int, b)
            label = f"PR {CLASS_NAMES.get(int(c), int(c))} {s:.2f}"
            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(image, label, (x1, max(40, y1 + 18)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    return image

def show_mined_examples(model, examples_df, title, conf=0.25, imgsz=512, n=9, cols=3):
    if len(examples_df) == 0:
        print(f"No examples for {title}")
        return

    examples_df = examples_df.head(n)
    rows = math.ceil(len(examples_df) / cols)
    plt.figure(figsize=(5 * cols, 4.5 * rows))

    for i, (_, row) in enumerate(examples_df.iterrows(), 1):
        pred = model.predict(
            source=str(row["image_path"]),
            conf=conf,
            imgsz=imgsz,
            device=0 if torch.cuda.is_available() else "cpu",
            verbose=False,
        )[0]

        vis = draw_gt_and_pred(row["image_path"], pred, row["label_path"])
        plt.subplot(rows, cols, i)
        plt.imshow(vis)
        plt.title(
            f'{row["image_name"]}\nfp={row["fp_count"]} fn={row["fn_count"]} '
            f'gt={row["gt_count"]} pred={row["pred_count"]}'
        )
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
RUN_ERROR_MINING = False

# Change this to the best.pt generated by your training run.
MODEL_PATH = "/kaggle/working/runs/detect/roboflow_fire_smoke_yolo26n/weights/best.pt"
ERROR_SPLIT = "test" if "test" in SPLITS else ("valid" if "valid" in SPLITS else SPLITS[-1])
CONF_THRESHOLD = 0.25
MATCH_IOU = 0.5
ERROR_IMGSZ = 512
ERROR_BATCH_SIZE = 32

if RUN_ERROR_MINING:
    error_model = YOLO(MODEL_PATH)
    error_df, top_fp, top_fn, top_both, error_dir = run_error_mining(
        error_model,
        split_name=ERROR_SPLIT,
        conf=CONF_THRESHOLD,
        imgsz=ERROR_IMGSZ,
        batch_size=ERROR_BATCH_SIZE,
        iou_threshold=MATCH_IOU,
    )

    print("Error manifests saved to:", error_dir)
    display(error_df[["fp_count", "fn_count", "gt_count", "pred_count"]].describe())
    display(top_fp.head(10))
    display(top_fn.head(10))

    show_mined_examples(error_model, top_fp, "Top False Positive Images", conf=CONF_THRESHOLD, imgsz=ERROR_IMGSZ, n=9, cols=3)
    show_mined_examples(error_model, top_fn, "Top False Negative Images", conf=CONF_THRESHOLD, imgsz=ERROR_IMGSZ, n=9, cols=3)
    show_mined_examples(error_model, top_both, "Top Mixed Error Images", conf=CONF_THRESHOLD, imgsz=ERROR_IMGSZ, n=9, cols=3)